# MLP-Trainingsskript (mit Baseline und Dropout)

`dropout_rate = 0.1` deaktiviert beim Training zufällig 10 % der Aktivierungen
nach der ersten versteckten Schicht. Mit `0.0` ist Dropout ausgeschaltet.
Bei diesem kleinen Netz zunächst niedrige Raten wie `0.1` oder `0.2` ausprobieren.
Dropout kann Overfitting reduzieren, garantiert aber keinen niedrigeren MAPE.
Für die Auswahl der Rate eine separate Validierungsmenge verwenden;
die Testmenge für die abschließende Bewertung reservieren.
`model.train()` aktiviert Dropout, `model.eval()` deaktiviert es für Vorhersagen.

In [16]:
import kagglehub
import pandas
import torch
import matplotlib.pyplot as plt
import numpy

from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PowerTransformer
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

# 1. Datensatz runterladen und mit Pandas einlesen
path = kagglehub.dataset_download("mchilamwar/predict-concrete-strength")
df = pandas.read_csv(f"{path}/ConcreteStrengthData.csv")
df = df.rename(columns={"CementComponent ": "CementComponent"})

# 2. Definiere, was Input x und Output y für das Modell sein soll
zielspalte = "Strength"
x = df.drop( zielspalte, axis="columns")
y = df[ [zielspalte] ]

# 3. Train-Test-Split
x_train, x_test, y_train, y_test = train_test_split(x,y,
                                                    test_size=0.2,
                                                    random_state=123
                                                   )
# 4. Skalierung der Daten
x_scaler = StandardScaler()
y_scaler = StandardScaler()
x_train_scaled = x_scaler.fit_transform(x_train)
y_train_scaled = y_scaler.fit_transform(y_train)

# 5. Erzeuge jetzt aus den Daten Tensoren
x_train_scaled_tensor = torch.tensor(x_train_scaled, dtype=torch.float32)
y_train_scaled_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

nr_inputs = x_train.shape[-1]
nr_outputs = y_train.shape[-1]

# Dropout-Wahrscheinlichkeit: 0.0 = aus, zunächst 0.1 ausprobieren.
dropout_rate = 0.1

MAPEs = []
for run in range(10):

    print(run, end=" ")
    # Reproduzierbare Initialisierung für jeden der zehn Läufe.
    torch.manual_seed(123 + run)

    # 6. Modell bauen
    N1 = 128
    N2 = 64
    model = nn.Sequential(
        nn.Linear(nr_inputs, N1),
        nn.ReLU(),
        nn.Dropout(p=dropout_rate),
        
        nn.Linear(N1, N2),
        nn.ReLU(),
        
        nn.Linear(N2, nr_outputs),
    )
    
    # 7. Modell trainieren
    loss_function = nn.MSELoss() # Mittler Quadratischer Fehler = Mean Squared Error
    #loss_function = nn.L1Loss() # MAE
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    model.train()
    losses = []
    for epoch in range(1000):        
        predictions = model( x_train_scaled_tensor )
        loss = loss_function(predictions, y_train_scaled_tensor)
        losses.append(loss.item())
            
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # 8. Das fertige MLP testen
    x_test_scaled = x_scaler.transform(x_test)
    x_test_scaled_tensor = torch.tensor(x_test_scaled, dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        predictions_test_scaled = model(x_test_scaled_tensor)
    predictions_test = y_scaler.inverse_transform(predictions_test_scaled.numpy())    
    y_pred = predictions_test.flatten()
    y_true = y_test["Strength"].values    
    MAPE = mean_absolute_percentage_error(y_true, y_pred)*100
    MAPEs.append( MAPE )

print()
print(MAPEs)
print(f"Dropout-Rate: {dropout_rate}")
print(f"Average MAPE : {numpy.mean(MAPEs):.1f}%")

# Abschließend ein Vergleich mit Standard-Baseline: MVLR

# 1. Trainiere die Lineare Regression
base_model = LinearRegression().fit(x_train_scaled, y_train_scaled)

# 2. Teste das Modell mit den Testdaten
y_pred_scaled = base_model.predict( x_test_scaled )
y_pred = y_scaler.inverse_transform( y_pred_scaled )

# 3. Berechne den MAPE der LR auf den Testdaten
y_true = y_test["Strength"].values
MAPE = mean_absolute_percentage_error(y_true, y_pred.flatten())*100
print(f"MVLR MAPE: {MAPE:.1f}%")

# Modell und Skalierer abspeichern
import pickle
datei = open("mlp_betonstaerken_regressor.pkl", "wb")
pickle.dump(x_scaler, datei)
pickle.dump(model, datei)
pickle.dump(y_scaler, datei)
datei.close()

0 1 2 3 4 5 6 7 8 9 
[9.881184117668084, 10.019276832510362, 9.633089373721441, 9.115607523494953, 9.738299638206328, 10.225304617168248, 9.52337330630204, 10.206352504442103, 9.415339078956302, 9.147247758060024]
Dropout-Rate: 0.1
Average MAPE : 9.7%
MVLR MAPE: 27.9%


# MLP-Inferenzskript

In [ ]:
x_test.head(3).to_csv("neue_beton_mischungen.csv", index=False)

In [ ]:
# 1. Modell und Skalierer wiederherstellen
import pickle
datei = open("mlp_betonstaerken_regressor.pkl", "rb")
x_scaler = pickle.load(datei)
model = pickle.load(datei)
y_scaler = pickle.load(datei)
datei.close()

# 2. Neue Betonmischungen einlesen
import pandas
df = pandas.read_csv("neue_beton_mischungen.csv")

# 3. Daten skalieren, durchs Modell, Modelloutput zurück skalieren
import torch
x_scaled = x_scaler.transform(df)
x_scaled_tensor = torch.tensor(x_scaled, dtype=torch.float32)
model.eval()  # Dropout auch nach dem Laden explizit ausschalten.
with torch.no_grad():
    predictions_scaled_tensor = model(x_scaled_tensor)
y_pred = y_scaler.inverse_transform( predictions_scaled_tensor.numpy() )

# 4. Prädizierte Betonstärken in Tabelle schreiben
df["PredStrength"] = y_pred
df

# 3. Vergleich mit weiteren Regressionsverfahren

Zuerst den Trainingsabschnitt ausführen. Alle fünf Verfahren verwenden exakt
`x_train_scaled`, `y_train_scaled` und `x_test_scaled` aus diesem Abschnitt:
Es gibt keinen neuen Split und kein erneutes Anpassen der StandardScaler.
Auch die Zielwerte bleiben beim Training standardisiert. Vor der Bewertung
werden die Vorhersagen mit `y_scaler` in die ursprüngliche Einheit zurückgerechnet.
Der MAPE wird auf denselben Testfällen berechnet und in Prozent angegeben.

Dies ist ein erster Vergleich mit den unten angegebenen Einstellungen, ohne
Hyperparameteroptimierung. Kleinere MAPE-Werte sind besser. Für späteres Tuning
nur die Trainingsdaten (z. B. mit Kreuzvalidierung) verwenden.


In [18]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Vorhandene skalierte Daten unverändert übernehmen; sklearn erwartet y als 1D-Array.
y_train_vergleich = y_train_scaled.ravel()
y_test_vergleich = y_test["Strength"].to_numpy()

vergleichsmodelle = {
    "DecisionTreeRegressor": DecisionTreeRegressor(random_state=123),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=100, random_state=123, n_jobs=-1
    ),
    "KNeighborsRegressor": KNeighborsRegressor(n_neighbors=5),
    "SVR": SVR(kernel="rbf", C=1.0, epsilon=0.1),
    "GradientBoostingRegressor": GradientBoostingRegressor(
        n_estimators=100, learning_rate=0.1, max_depth=3, random_state=123
    ),
}

vergleichsergebnisse = []
for name, regressor in vergleichsmodelle.items():
    regressor.fit(x_train_scaled, y_train_vergleich)
    vorhersagen_scaled = regressor.predict(x_test_scaled)
    vorhersagen = y_scaler.inverse_transform(
        vorhersagen_scaled.reshape(-1, 1)
    ).ravel()
    vergleichsergebnisse.append({
        "Modell": name,
        "MAPE (%)": mean_absolute_percentage_error(
            y_test_vergleich, vorhersagen
        ) * 100,
        "MAE": mean_absolute_error(y_test_vergleich, vorhersagen),
        "R²": r2_score(y_test_vergleich, vorhersagen),
    })

# Numerische Werte behalten; nur die Anzeige runden.
ergebnisse_df = (
    pandas.DataFrame(vergleichsergebnisse)
    .sort_values("MAPE (%)", ascending=True)
    .reset_index(drop=True)
)
ergebnisse_df.round({"MAPE (%)": 2, "MAE": 2, "R²": 3})


,Modell,MAPE (%),MAE,R²
0,RandomForestRegressor,10.39,3.42,0.893
1,GradientBoostingRegressor,11.22,3.71,0.897
2,DecisionTreeRegressor,12.74,4.04,0.833
3,SVR,14.03,4.54,0.843
4,KNeighborsRegressor,20.42,6.09,0.753
